SECTION 1

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "preprocessed_demand_forecasting_data.csv",
    parse_dates=['Date']
)

In [3]:
print(df.shape)
print(df.head())

(75860, 25)
        Date Store ID Product ID     Category Region  Inventory Level  Price  \
0 2022-01-02     S003      P0001         Toys   East               73  15.76   
1 2022-01-02     S004      P0001    Groceries   West              130  12.67   
2 2022-01-02     S005      P0001    Groceries  North              107  11.70   
3 2022-01-03     S001      P0001  Electronics  North              274  68.55   
4 2022-01-03     S002      P0001    Groceries  South                0  84.32   

   Discount Weather Condition  Promotion  ...  month  year  weekday  \
0         5            Cloudy          0  ...      1  2022        6   
1        10             Sunny          0  ...      1  2022        6   
2         0             Snowy          0  ...      1  2022        6   
3        15             Snowy          1  ...      1  2022        0   
4        20            Cloudy          1  ...      1  2022        0   

   weekofyear  is_weekend  Discounted Price  lag_1  lag_7  rolling_mean_7  \
0  

In [4]:
target = "Demand"

In [5]:
features = [col for col in df.columns if col not in ["Date", "Demand"]]

In [6]:
X = df[features]
y = df[target]

In [7]:
print("\nFeatures:")
print(features)

print("\nTarget:")
print(target)


Features:
['Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'day', 'month', 'year', 'weekday', 'weekofyear', 'is_weekend', 'Discounted Price', 'lag_1', 'lag_7', 'rolling_mean_7', 'discount_bin']

Target:
Demand


SECTION 2 - Label Encoding

In [8]:
from sklearn.preprocessing import LabelEncoder

# Copy X to avoid modifying original
X_encoded = X.copy()

# Label Encoding

label_cols = ["Product ID", "Store ID"]

label_encoders = {}

for col in label_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col])
    label_encoders[col] = le

In [9]:
# One-Hot Encoding

onehot_cols = [
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality",
    "discount_bin"
]

X_encoded = pd.get_dummies(
    X_encoded,
    columns=onehot_cols,
    drop_first=True
)

In [10]:
# Convert boolean columns to 0/1
bool_cols = X_encoded.select_dtypes(include='bool').columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)

In [11]:
# Clean column names for XGBoost
X_encoded.columns = X_encoded.columns.str.replace('[', '', regex=False)
X_encoded.columns = X_encoded.columns.str.replace(']', '', regex=False)
X_encoded.columns = X_encoded.columns.str.replace('<', '', regex=False)

In [12]:
print(X_encoded.shape)

(75860, 33)


In [13]:
print(X_encoded.head())

   Store ID  Product ID  Inventory Level  Price  Discount  Promotion  \
0         2           0               73  15.76         5          0   
1         3           0              130  12.67        10          0   
2         4           0              107  11.70         0          0   
3         0           0              274  68.55        15          1   
4         1           0                0  84.32        20          1   

   Competitor Pricing  Epidemic  day  month  ...  Region_South  Region_West  \
0               16.81         0    2      1  ...             0            0   
1               11.95         0    2      1  ...             0            1   
2               10.06         0    2      1  ...             0            0   
3               80.73         0    3      1  ...             0            0   
4               92.33         0    3      1  ...             1            0   

   Weather Condition_Rainy  Weather Condition_Snowy  Weather Condition_Sunny  \
0           

Section 3 - Time Aware

In [14]:
# Time-Aware Train/Test Split
split_date = "2023-09-01"

# Train set
X_train = X_encoded[df["Date"] < split_date]
y_train = y[df["Date"] < split_date]

# Test set
X_test = X_encoded[df["Date"] >= split_date]
y_test = y[df["Date"] >= split_date]

In [15]:
print("Training set shape:", X_train.shape)

Training set shape: (60660, 33)


In [16]:
print("Testing set shape:", X_test.shape)

Testing set shape: (15200, 33)


In [17]:
print("\nTrain Date Range:")
print(df[df["Date"] < split_date]["Date"].min(),
      "to",
      df[df["Date"] < split_date]["Date"].max())


Train Date Range:
2022-01-02 00:00:00 to 2023-08-31 00:00:00


In [18]:
print("\nTest Date Range:")
print(df[df["Date"] >= split_date]["Date"].min(),
      "to",
      df[df["Date"] >= split_date]["Date"].max())


Test Date Range:
2023-09-01 00:00:00 to 2024-01-30 00:00:00


SECTION 4 - Linear Regression

In [19]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [20]:
# Linear Regression Model:

lr_model = LinearRegression()

# Train model
lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test)

# Evaluation
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

In [21]:
print("Linear Regression Results")
print("MAE :", round(mae_lr, 2))
print("RMSE:", round(rmse_lr, 2))

Linear Regression Results
MAE : 24.29
RMSE: 31.04


SECTION 5 — RANDOM FOREST REGRESSOR

In [22]:
from sklearn.ensemble import RandomForestRegressor

In [23]:
# Random Forest Regressor:

# Initialize model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train model
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

In [24]:
# Evaluation
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

In [25]:
print("Random Forest Results")
print("MAE :", round(mae_rf, 2))
print("RMSE:", round(rmse_rf, 2))

Random Forest Results
MAE : 17.43
RMSE: 24.19


SECTION 6 — BASIC XGBOOST MODEL

In [26]:
from xgboost import XGBRegressor

In [27]:
# XGBoost Regressor:

# Initialize model
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

In [28]:
# Train model
xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)

In [29]:
# Evaluation
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

In [30]:
print("XGBoost Results")
print("MAE :", round(mae_xgb, 2))
print("RMSE:", round(rmse_xgb, 2))

XGBoost Results
MAE : 18.86
RMSE: 25.08


SECTION 7 — XGBOOST HYPERPARAMETER TUNING

In [31]:
from sklearn.model_selection import RandomizedSearchCV

In [32]:
# Hyperparameter Grid:

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

In [33]:
# Base XGBoost Model

xgb_base = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

In [34]:
# Randomized Search:

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=10,
    scoring='neg_mean_absolute_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

In [35]:
# Train search
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [4, 6, ...], 'n_estimators': [100, 200], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can

In [36]:
# Best model
best_xgb = random_search.best_estimator_

# Predictions
y_pred_tuned = best_xgb.predict(X_test)

In [37]:
# Evaluation
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

In [38]:
# Results
print("Tuned XGBoost Results")
print("MAE :", round(mae_tuned, 2))
print("RMSE:", round(rmse_tuned, 2))

Tuned XGBoost Results
MAE : 23.16
RMSE: 29.88


In [39]:
print("\nBest Parameters:")
print(random_search.best_params_)


Best Parameters:
{'subsample': 0.8, 'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


SECTION 7B — TimeSeriesSplit XGBoost Tuning

In [40]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import RandomizedSearchCV

In [41]:
# TimeSeriesSplit:

tscv = TimeSeriesSplit(n_splits=3)

# Parameter Grid:

param_grid_ts = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Base Model

xgb_ts = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

# Randomized Search

random_search_ts = RandomizedSearchCV(
    estimator=xgb_ts,
    param_distributions=param_grid_ts,
    n_iter=10,
    scoring='neg_mean_absolute_error',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Train model
random_search_ts.fit(X_train, y_train)

# Best model
best_xgb_ts = random_search_ts.best_estimator_

# Predictions
y_pred_ts = best_xgb_ts.predict(X_test)

# Evaluation
mae_ts = mean_absolute_error(y_test, y_pred_ts)
rmse_ts = np.sqrt(mean_squared_error(y_test, y_pred_ts))

# Results
print("TimeSeriesSplit Tuned XGBoost Results")
print("MAE :", round(mae_ts, 2))
print("RMSE:", round(rmse_ts, 2))

print("\nBest Parameters:")
print(random_search_ts.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
TimeSeriesSplit Tuned XGBoost Results
MAE : 21.3
RMSE: 27.9

Best Parameters:
{'subsample': 1.0, 'n_estimators': 100, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
